# 🦾 Fine-tune ACT on our SO101 wrist camera

This notebook **fine-tunes** the pretrained [`TakuyaHiraoka/act_so101_pick_diverse_objects`](https://huggingface.co/TakuyaHiraoka/act_so101_pick_diverse_objects) ACT policy on our own demos, so the policy adapts to **our wrist camera** (mounting, lens, lighting, table) instead of learning to pick from scratch.

### How this differs from `Training Act Notebook.ipynb`
| | Training notebook | This notebook |
|---|---|---|
| Start point | `--policy.type=act` (random transformer, ImageNet ResNet18) | `--policy.path=TakuyaHiraoka/...` (already knows how to pick, 360k steps) |
| Data needed | 50+ episodes | ~10 episodes works for camera adaptation |
| Steps | 20k+ | ~8k (≈11 epochs of 11k frames at batch 16) |
| Normalization | from our dataset | **recomputed from our dataset** (lerobot does this automatically when fine-tuning) |
| Augmentation | off | **on** (color jitter + small affine) to generalize across lighting / slight camera shifts |

### Compatibility (verified)
Base model inputs: `observation.images.wrist` 3×480×640, `observation.state` 6, `action` 6, 30 fps.
Our dataset `so101_pick_test1`: 10 episodes, 11,158 frames, same keys and shapes → no `rename_map` needed.

### ⚙️ Before running
On the Mac, upload the local recording to the Hub (it is not on the Hub yet):
```bash
conda activate lerobot
python training/push_dataset.py \
  --root ~/.cache/huggingface/lerobot/Jovian-Dsouza/so101_pick_test1_20261010_213115 \
  --repo-id Jovian-Dsouza/so101_pick_test1
```
Then run this notebook on a GPU runtime (T4 is fine; A100 ≈ 15 min).

## 1. Settings
Edit these, everything below reads from them.

In [ ]:
BASE_POLICY   = "TakuyaHiraoka/act_so101_pick_diverse_objects"   # checkpoint to fine-tune
DATASET_REPO  = "Jovian-Dsouza/so101_pick_test1"                 # our 10 wrist-camera episodes
POLICY_REPO   = "Jovian-Dsouza/act_so101_pick_wrist_ft"          # where the fine-tuned model is pushed
JOB_NAME      = "act_so101_pick_wrist_ft"
OUTPUT_DIR    = f"outputs/train/{JOB_NAME}"

DEVICE        = "cuda"   # "mps" on Apple Silicon, "cpu" otherwise
BATCH_SIZE    = 16       # drop to 8 if you hit CUDA OOM
STEPS         = 8_000    # ~11 epochs on 11k frames; small data overfits past this
SAVE_FREQ     = 2_000    # keep intermediate checkpoints to compare on the robot
AUGMENT       = True     # image augmentation helps with lighting / camera differences
WANDB         = False

## 2. Install LeRobot

In [ ]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install -y ffmpeg
!cd lerobot && pip install -e ".[train, dataset]"

## 3. Logins
HF token needs **write** access (to read the private dataset and push the model). W&B is optional.

In [ ]:
!hf auth login
# !wandb login   # only if WANDB = True

## 4. Sanity check: dataset matches the base policy
Fails fast here instead of 10 minutes into training if a camera key or shape is wrong.

In [ ]:
import json
from huggingface_hub import hf_hub_download

policy_cfg = json.load(open(hf_hub_download(BASE_POLICY, "config.json")))
info = json.load(open(hf_hub_download(DATASET_REPO, "meta/info.json", repo_type="dataset")))

expected = {k: v["shape"] for k, v in {**policy_cfg["input_features"], **policy_cfg["output_features"]}.items()}
features = info["features"]

def chw(key, shape):
    return [shape[2], shape[0], shape[1]] if key.startswith("observation.images.") else shape

problems = [
    f"{k}: missing" if k not in features else f"{k}: dataset {chw(k, features[k]['shape'])} != policy {s}"
    for k, s in expected.items()
    if k not in features or list(chw(k, features[k]["shape"])) != list(s)
]
print(f"Dataset: {info['total_episodes']} episodes, {info['total_frames']} frames @ {info['fps']} fps")
assert not problems, "Incompatible dataset:\n" + "\n".join(problems)
print("✅ Dataset features match", BASE_POLICY)

## 5. Fine-tune

Key flags:
- `--policy.path` loads the pretrained weights **and** its config (chunk_size 100, resnet18, lr 1e-5). Don't pass `--policy.type` together with it.
- Normalization stats are replaced with stats from our dataset, so our joint calibration/ranges are handled.
- `--dataset.image_transforms.enable` turns on random brightness/contrast/saturation/hue/sharpness/affine; this is the main lever for "understanding our camera" with only 10 episodes.
- The policy is pushed to `POLICY_REPO` (private) at the end.

In [ ]:
!lerobot-train \
  --policy.path={BASE_POLICY} \
  --dataset.repo_id={DATASET_REPO} \
  --dataset.image_transforms.enable={str(AUGMENT).lower()} \
  --policy.device={DEVICE} \
  --policy.repo_id={POLICY_REPO} \
  --policy.push_to_hub=true \
  --policy.private=true \
  --batch_size={BATCH_SIZE} \
  --steps={STEPS} \
  --save_freq={SAVE_FREQ} \
  --log_freq=100 \
  --output_dir={OUTPUT_DIR} \
  --job_name={JOB_NAME} \
  --wandb.enable={str(WANDB).lower()}

## 6. (Optional) Push an earlier checkpoint

With only 10 episodes the last checkpoint is not always the best on the real robot. If an earlier one behaves better, push it to its own repo:

In [ ]:
CHECKPOINT = "004000"   # one of the folders in OUTPUT_DIR/checkpoints
!ls {OUTPUT_DIR}/checkpoints
!hf upload {POLICY_REPO}_{CHECKPOINT} {OUTPUT_DIR}/checkpoints/{CHECKPOINT}/pretrained_model . --private

## 7. Run it on the robot

```bash
lerobot-record \
  --robot.type=so101_follower \
  --robot.port=/dev/tty.wchusbserial5A7B2891531 \
  --robot.id=so_follower \
  --robot.cameras="{ wrist: {type: opencv, index_or_path: <SAME_INDEX_AS_RECORDING>, width: 640, height: 480, fps: 30}}" \
  --dataset.repo_id=Jovian-Dsouza/eval_act_so101_pick_wrist_ft \
  --dataset.single_task="fine tunning" \
  --dataset.num_episodes=5 \
  --dataset.push_to_hub=false \
  --policy.path=Jovian-Dsouza/act_so101_pick_wrist_ft
```
Use the **same camera index/resolution as during recording**, and keep `single_task` the same as the dataset's task string.

### If it still struggles
- Record 10-20 more episodes with the object in different positions, then re-run (more data beats more steps).
- Overfitting (smooth in replay but jittery live) → use an earlier checkpoint or lower `STEPS`.
- Underfitting (still ignores the object) → raise `STEPS` to 15k.